In [ ]:
import pandas as pd

# generate by ChatGPT, with some manual edits
with open("data_asg1.txt", encoding="utf-8") as f:
    texts = f.readlines()

with open("labels_asg1.txt") as f:
    labels = f.read().split()

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df.head()

In [ ]:
# 2) Word-based embeddings (Wikipedia skip-gram 300d)

import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# 2.0) Load tweets and labels into a DataFrame
with open("data_asg1.txt", encoding="utf-8") as f:
    texts = f.readlines()

with open("labels_asg1.txt") as f:
    labels = f.read().split()

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

print("Number of samples:", len(df))

In [ ]:
# 2.1) Load the pretrained Wikipedia embeddings from model.txt (word2vec text format)
model_path = "model.txt"  # model.txt is in the same folder as HWK1.ipynb
w2v = KeyedVectors.load_word2vec_format(model_path, binary=False)

EMBED_DIM = w2v.vector_size  # should be 300
print("Embedding dim:", EMBED_DIM)
print("Vocab size:", len(w2v.key_to_index))

In [ ]:
# 2.2) Convert each tweet into an averaged word embedding
def sentence_to_vec(sentence, w2v, dim):
    # simple whitespace tokenization; can be improved later
    words = sentence.split()
    vectors = [w2v[word] for word in words if word in w2v.key_to_index]
    # if no word has a pretrained vector, return a zero vector
    if not vectors:
        return np.zeros(dim, dtype="float32")
    # average all word vectors in the tweet
    return np.mean(vectors, axis=0)

X_wiki = np.vstack([
    sentence_to_vec(text, w2v, EMBED_DIM)
    for text in df["text"].values
])
y = df["label"].values

print("X_wiki shape:", X_wiki.shape)  # (num_samples, 300)


In [ ]:
# 2.3) Apply PCA on sentence embeddings (300 -> 50 dimensions)
pca = PCA(n_components=50, random_state=42)
X_wiki_pca = pca.fit_transform(X_wiki)


In [ ]:
# 2.4) Define a helper to run 5-fold CV and compute metrics
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_scores(X, y):
    accs, f1s, macro_f1s = [], [], []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf = LinearSVC()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred, pos_label="1"))
        macro_f1s.append(f1_score(y_test, y_pred, average="macro"))
    return np.mean(accs), np.mean(f1s), np.mean(macro_f1s)


In [ ]:
# 2.5) Compare performance before and after PCA
print("Wiki 300d (no PCA):", cv_scores(X_wiki, y))
print("Wiki 300d + PCA 50d:", cv_scores(X_wiki_pca, y))


In [ ]:
# 2.6)t-SNE visualization for Wikipedia sentence embeddings

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Run t-SNE on the 300d sentence embeddings
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30,
    learning_rate=200,
    init="random"
)
X_wiki_tsne = tsne.fit_transform(X_wiki)

# Plot, color by label (0 = non-irony, 1 = irony)
colors = ["blue" if label == "0" else "red" for label in y]

plt.figure(figsize=(6, 6))
plt.scatter(X_wiki_tsne[:, 0], X_wiki_tsne[:, 1],
            c=colors, alpha=0.5, s=5)
plt.title("t-SNE of Wikipedia 300d tweet embeddings")
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")
plt.tight_layout()
plt.show()
